# Level 3 template analysis

Level 3 asks counterfactual questions: given a baseline run and an event injected into it,
what would have happened. Every item pairs a context window taken from the baseline episode
with rows from the counterfactual episode after the event onset.

This notebook records the defects found in the released level 3 templates and the fix applied
to each. It is the level 3 counterpart to `real_analysis.ipynb`. The unit here is the
**template defect**, not the individual item: every defect below was systematic, affecting
either all items of a template or a measurable fraction of them, so per-item solving would
have missed the pattern.

The released `factorybench_qa/level_3/*.jsonl` still contains the items produced by defects 4
to 10. Those fixes live in the generators and take effect when level 3 is regenerated.
Defects 1 to 3 were backfilled onto the release directly.


## Quick reference, one line per defect

| # | Template | Defect | Scale | Status |
|---|---|---|---|---|
| [1](#defect-1) | all L3 | Question named an event timestep outside the window shown | 1,937 / 2,939 items | Fixed, backfilled |
| [2](#defect-2) | L3.1 | Option segments appeared verbatim inside their own context | 696 options, 304 contexts | Fixed, backfilled |
| [3](#defect-3) | L3.1 | Options described ~98 channels, context showed ~19 | 1,317 items | Fixed, backfilled |
| [4](#defect-4) | L3.2, L3.3 | Options asserted TCP tracking with no measured TCP shown | 741 items, 0% answerable | Fixed in generator |
| [5](#defect-5) | L3.2, L3.3 | Three statements had no evaluator, scored as confident False | 100% False regardless of data | Fixed in generator |
| [6](#defect-6) | L3.2, L3.3 | Thresholds above 1 silently clipped to 0.99 | every degree and Nm statement | Fixed in generator |
| [7](#defect-7) | L3.2, L3.3 | No fitted thresholds at all | whole corpus | Fixed, table fitted |
| [8](#defect-8) | L3.2, L3.3 | Statements compared against a single baseline row | 4 statement pairs | Fixed in generator |
| [9](#defect-9) | L3.2, L3.3 | Two options in one item could be exact negations | possible on every item | Fixed in generator |
| [10](#defect-10) | L3.2, L3.3 | Statements no threshold can calibrate stayed in the pool | 8 statements | Excluded |

Headline effect on L3.3: guessing the per-position majority scored **0.700** against a chance
rate of 0.500 and now scores **0.577**. A solver that recognises the statement and ignores the
time series entirely scored **86.4%** on held-out option slots and now scores **77.3%**.


<a id="defect-1"></a>
## Defect 1. The event was outside the window shown

Every level 3 question names the timestep at which the counterfactual event occurs, for
example "Given the counterfactual scenario where a collision foam object occurs at timestep
1616 ms". That number was written from the raw episode clock, while the context window is
renormalised so its first row is `t=0`. The two clocks disagree by however far into the
episode the window starts, so the timestep in the question routinely pointed outside the rows
the model was given, sometimes far outside.

The model was being asked to reason about an event it could not locate.

**Cause.** `level3.py` built the event description from the raw row index, before
`normalize_timestamps` rebased the window.

**Fix.** The onset is now located with `find_event_onset_index` on the same normalised clock
the context is written in, so the stated timestep is by construction one the model can see.

**Result.** 1,937 of 2,939 released items carried the wrong number and were rewritten by
`scripts/backfill_level3_onset_timestep.py`. After the backfill, 0 of 2,939 fall outside their
own window. This one was safe to repair in place: the context and the answer are unchanged,
only the sentence was wrong.


<a id="defect-2"></a>
## Defect 2. L3.1 showed part of the answer inside the question

L3.1 gives four signal segments and asks for the order in which they appear as the event
manifests. That is only a question about the future if the segments lie outside the window
already shown. At level 3 the baseline and counterfactual episodes are identical until they
diverge, so segments sampled just after the onset reproduce rows the model can already read,
and the ordering can be recovered by matching values instead of reasoning about how a
disturbance propagates.

**Cause.** Segments were sampled from `post_event_rows`, which starts at the onset and
therefore overlaps the tail of the context window.

**Fix.** `rows_strictly_after_context` keeps only rows whose timestamp is strictly greater
than the last timestamp in the context, so a segment can never be one the model has seen.

**Result.** 13% of option segments appeared verbatim inside their own context. 304 contexts
were trimmed and the 696 overlapping options went to 0, applied by
`scripts/trim_ranking_context_overlap.py`.


<a id="defect-3"></a>
## Defect 3. L3.1 options and context described different signals

`build_context` trims the context to the template's `important_features`, but the option
segments were encoded from the full row. The result was a question defined over two different
feature sets: the context showed about 19 channels while the segments to be ordered carried
about 98, and 79 of those appeared in no acronym mapping anywhere in the item, so the model
had no way to know what those numbers referred to.

**Cause.** The segment encoder ran on the raw row while the context encoder ran on the
filtered row.

**Fix.** `filter_rows_to_template_features` applies the same restriction to the segments, so
both halves of the question describe the same signals.

**Result.** 1,317 items realigned, median channels per segment down from 98 to 18, applied by
`scripts/align_ranking_option_features.py`.


<a id="defect-4"></a>
## Defect 4. Options asserted TCP tracking with no TCP measurement shown

`mc_015` and `mc_016` claim that command and measured TCP motion stay aligned, or become
misaligned. Judging either needs the commanded TCP channels **and** the measured ones.

| | items | has command side | has measured side | has both |
|---|---|---|---|---|
| L3.3 | 391 | 100.0% | 0.0% | 0.0% |
| L3.2 | 350 | 100.0% | 0.0% | 0.0% |
| L2.2 | 74 | 0.0% | 0.0% | 0.0% |
| L2.3 | 51 | 0.0% | 0.0% | 0.0% |

Not one of the 866 items was answerable. The clean 100% / 0% split is the signature of the
second cause below.

**Cause, two of them.**

1. The availability filter ran against the whole episode rather than the channels the context
   exposes. An episode carries all 24 TCP channels, so the option passed even on L2.2 and
   L2.3, whose templates list 30 features and not one of them a TCP channel.
2. `required_features` was read as a list of alternatives. The command family alone satisfied
   a statement that compares command against measurement.

**Fix.** The filter now runs on context-restricted rows. `required_features` supports `a|b`
alternation and a per-option `require` field, and `mc_015` and `mc_016` now declare that they
need a command channel and a measured one, either of which may be a pose or a speed.

**Result.** On L2.2 and L2.3, where the context can never show TCP, the option is correctly
dropped and appears on 0 items instead of 125 unanswerable ones. On L3.2 and L3.3, where the
context does carry TCP, it appears on 49 of 120 sampled items and all 49 show both sides.


<a id="defect-5"></a>
## Defect 5. Three statements had no evaluator and were scored False

`mc_030` (a joint sweeps more than N degrees), `mc_031` (the most active joint accumulates
more than N degrees) and `mc_032` (commanded torque peak to peak) were handled by the level 2
evaluator but had no branch in the level 3 one. `evaluate_mc_statement` fell through and
returned `None`, and the option builder turns `None` into a confident **False**.

So two statements were False on 100% of released L3.3 items regardless of what the robot did.
No threshold could have moved them, because the threshold was never consulted. This is why
the first calibration pass changed so little: it was tuning a value nothing read.

**Fix.** The three handlers were ported from the level 2 evaluator, together with the
unknown-versus-false guard level 2 already had. A robot that records no speed channel now
yields `None`, which the builder treats as not offerable, instead of asserting that no drop
occurred. That guard alone covers five statements: `mc_003`, `mc_004`, `mc_005`, `mc_012`
and `mc_019`.

**Result.** The two excursion statements went from 0.00 to 0.51 and 0.53.


<a id="defect-6"></a>
## Defect 6. Any threshold above 1 was silently clipped

`_sample_ratio` draws a per-item threshold around a fitted centre. Level 2 had already removed
its `max_value=0.99` default after finding that it clipped any centre above 1; level 3 still
carried the old default.

Every threshold expressed in degrees or Nm has a centre far above 1: `joint_excursion_deg` at
67, `joint_path_deg` at 104.4, `torque_p2p_nm` at 106.6. All were being clamped to 0.99, so
the statement was satisfied on essentially every window and stopped depending on its
threshold.

**Fix.** `max_value` now defaults to no upper bound. The two callers that genuinely need a
sub-1 cap, the coverage and axis-ratio knobs, pass it explicitly and are unaffected.


<a id="defect-7"></a>
## Defect 7. Level 3 had no fitted thresholds at all

Level 2 keeps a `PER_ROBOT_THRESHOLDS` table fitted against the windows its generator actually
samples. Level 3 shipped with hand-picked defaults and no table. Combined with defects 5 and 6
this left the per-position rates near 0.30 instead of 0.50:

| released L3.3 | |
|---|---|
| P(A=T) | 0.307 |
| P(B=T) | 0.317 |
| P(C=T) | 0.307 |
| P(D=T) | 0.269 |
| majority-class guess | 0.700 |
| top answer string | FFFF at 15.7% |

Answering all-False scored 0.700 against a chance rate of 0.500.

**Fix.** `scripts/calibrate_l3_tmpl3_thresholds.py` sweeps each threshold over the windows the
generator samples and picks the value whose true rate is closest to 50%. It evaluates through
`evaluate_mc_statement` itself rather than reimplementing the statistic, so the calibration
cannot drift from the scorer.

It builds windows the way **level 3** builds them, which is not the way level 2 does: the
context comes from the baseline episode around the onset and the judged rows come from the
counterfactual episode from the onset onward. Calibrating on level 2 windows would have fitted
the wrong distribution. Per-source tables were fitted for `factorywave_ur3` and
`factorywave_kuka`.


<a id="defect-8"></a>
## Defect 8. Statements were measured against a single baseline row

This is the one that took longest to see, and I got it wrong first.

After calibration, ten of seventeen statement families were still outside [0.15, 0.85] and the
sweep could not move them: they jumped from almost always false to almost always true with
nothing in between. I concluded the underlying quantity was bimodal and reported it as
structural, something only a redefinition of the statements could fix.

That was wrong. The quantities have plenty of spread. What killed them was the **comparison**.
Each statement compared its post-event value against `baseline`, a single instantaneous row,
the last sample of the context. When that sample sits near zero the threshold `(1 +/- r) * pre`
either explodes or collapses, and the statement stops depending on the threshold at all.

Checking the distribution of the replacement quantities over 400 sampled windows settled it:

| candidate statistic | p10 | median | p90 | median splits |
|---|---|---|---|---|
| min joint mean\|post\| / mean\|pre\| | 0.076 | 0.572 | 1.70 | yes |
| max (peak - final) / peak | 0.708 | 0.798 | 0.930 | yes |
| mean post `robot_current` | 0.688 | 0.750 | 0.791 | yes |
| mean post TCP tracking error | 0.0011 | 0.0025 | 0.0131 | yes |
| max joint temperature range | 0 | **0** | 0.50 | no |

Level 2 had already reached the same conclusion for the tracking pair and moved to an absolute
post-event error. Its own comment records the reason: post/pre tracking error is so
concentrated on this corpus that only 20 of 78 windows flip at any threshold. The same
reasoning applies to four more pairs, which level 3 had never had applied to them.

**Fix.** Each pair now shares one threshold and the two statements are exact complements:

| pair | was judged on | now judged on |
|---|---|---|
| `mc_003` / `mc_004` | `min(post speed) <= (1-r) x baseline_row` | ratio of window means |
| `mc_012` | peak vs baseline row **and** tail vs peak | `(peak - final) / peak` |
| `mc_013` / `mc_014` | peak-to-peak vs baseline row | absolute mean `robot_current` |
| `mc_015` / `mc_016` | `post/pre` TCP error ratio | absolute mean post TCP error |

**Result.** `mc_003` went from 1.00 to 0.47, `mc_012` from 1.00 to 0.48, `mc_014` from 0.07 to
0.51, `mc_016` from 0.07 to 0.50, `mc_008` from 0.88 to 0.55.


<a id="defect-9"></a>
## Defect 9. Two options in one item could be exact negations

Making each pair share a threshold has a consequence: the two members become exact negations
of each other. Nothing stopped the builder drawing both into the same item, and when it did,
the answer at those two positions was always exactly one T whatever the episode did. A model
that notices the negation gets one of them for free.

Level 2 was already safe here, by accident rather than design: its `MC_STATEMENT_FAMILY` map
admits at most one statement per family, and all four pairs sit inside a family. Level 3 had
no such map.

**Fix.** `COMPLEMENT_PAIRS` names the four pairs and the selection loop admits a statement
only if it is not excluded, not already present, and not the negation of one already taken.
If fewer than four survive, the builder raises and the caller skips the item rather than
emitting a short answer string, which would break the length the scorer expects.

**Result.** 0 of 502 regenerated items contain a complement pair, and all 502 carry exactly
four options.


<a id="defect-10"></a>
## Defect 10. Statements no threshold can fix stayed in the pool

Eight statements could not be brought inside [0.15, 0.85] by any threshold on this corpus.
Keeping them means shipping options whose truth value is fixed regardless of the episode,
which is the defect the calibration exists to remove.

| statement | why |
|---|---|
| `mc_005` | stall condition never fires on this corpus, 0.00 |
| `mc_006`, `mc_007` | contact force: factorywave records no such channel |
| `mc_010`, `mc_011` | vibration: likewise absent |
| `mc_017`, `mc_018` | joint temperature is flat, median in-window range 0 |
| `mc_032` | commanded torque undeterminable on 88% of windows |

Six of the eight are data limitations rather than design errors: the channel is not recorded,
so nothing can be asked about it. The temperature pair is the only case where the channel
exists and still carries no usable signal.

**Fix.** They are named in `UNCALIBRATABLE_MC_IDS` and excluded from the level 3 pool. The
eleven that remain are enough: every regenerated item fills its four options.


<a id="results"></a>
## Results

From a 600-item regeneration at seed 33, compared against the release:

| L3.3 | released | after |
|---|---|---|
| P(A=T) | 0.307 | 0.451 |
| P(B=T) | 0.317 | 0.390 |
| P(C=T) | 0.307 | 0.403 |
| P(D=T) | 0.269 | 0.448 |
| majority-class guess | **0.700** | **0.577** |
| top answer string | FFFF at 15.7% | FFFF at 10.4% |
| reachable answer strings | 15/16 | 16/16 |
| statements outside [0.15, 0.85] | 12 of 17 | **0 of 11** |
| items with an exact-negation pair | not prevented | **0** |
| option slots missing their features | 741 | **0** |
| options per item | 4 | 4 |

A second measure, fitted on half the items and scored on the other half, asks what a solver
gets by recognising the statement and ignoring the time series entirely:

| held-out slot accuracy | released | after calibration | after defect 8 |
|---|---|---|---|
| answer the majority | 68.1% | 57.5% | 57.6% |
| recognise the statement family | 86.4% | 83.6% | **77.3%** |

For reference, level 2 template 3 scores 79.7% on the same held-out measure, so level 3 now
sits slightly ahead of the level it was being compared against.

### Both robots are covered

I claimed at one point that the no-complements rule had eliminated KUKA from L3.2 and L3.3,
and asserted it without measuring. It had not. KUKA has exactly five usable statements, and
since the selection loop skips a rejected candidate and keeps going rather than giving up,
four of them are always reachable.

| | items | P(A..D) | majority-class | options | complement pairs |
|---|---|---|---|---|---|
| kuka | 59 (11.8%) | 0.424 / 0.525 / 0.492 / 0.424 | 0.547 | 4 | 0 |
| ur3 | 443 (88.2%) | 0.463 / 0.470 / 0.497 / 0.533 | 0.526 | 4 | 0 |

KUKA sits at 11.8% against its 14.7% share of the corpus. The shortfall comes from KUKA
episodes failing other preconditions such as event onset and window length, not from the
option pool.


<a id="scoreboard"></a>
## Scoreboard

| Defect | Where it lived | Backfilled onto the release | Needs regeneration |
|---|---|---|---|
| 1. event outside the window | `level3.py` event description | yes, 1,937 items | |
| 2. options visible in context | `level3.py` segment sampling | yes, 304 contexts | |
| 3. options vs context features | `level3.py` segment encoding | yes, 1,317 items | |
| 4. TCP claim without measurement | `mc_availability.py`, option catalogue | | yes |
| 5. missing evaluators | `level3/mc_truth.py` | | yes |
| 6. threshold clipped at 0.99 | `level3.py` `_sample_ratio` | | yes |
| 7. no fitted thresholds | `level3/mc_truth.py` | | yes |
| 8. compared to one baseline row | `level3/mc_truth.py` | | yes |
| 9. exact-negation options | `level3.py` option selection | | yes |
| 10. uncalibratable statements | `level3/mc_truth.py` | | yes |

Defects 1 to 3 left the item intact and only its presentation wrong, so they were backfilled.
Defects 4 to 10 change which options appear and what the answer is, so the released items
carry them until level 3 is regenerated.

### What I would do differently

Two of these cost far more time than they should have, both for the same reason: I reported a
conclusion before measuring the thing the conclusion was about.

On defect 8 I established that no threshold could split ten statement families and concluded
the quantities themselves were bimodal, which would have meant redefining the statements.
Checking the distribution of the candidate quantities took one pass over 400 windows and
showed four of five had a clean median split. The problem was the comparison, not the signal.

On robot coverage I stated that KUKA had been excluded, reasoning from a ceiling calculation
rather than from the generated items. Counting the output showed KUKA present at close to its
corpus share the whole time.

The general lesson is that a ceiling calculation is a hypothesis, not a result. Both times the
measurement was cheap and I skipped it.
